# QA inference run record: LoRA + DeepSeek-R1-Distill-Llama-8B

This cleaned notebook copy preserves the source code and saved execution timestamps from the Q1 (300-question) inference run. Saved outputs were cleared to avoid redistributing benchmark text and generated answers. The original model mapping was verified by exact row-wise matching of all 300 generated answers against the corresponding model-specific Q1 evaluation CSV.

Set `QA_MODEL_PATH` to the locally available final adapter/checkpoint, `QA_TEST_CSV` to the prepared test CSV, and optionally `QA_OUTPUT_DIR` to choose an output directory. The private Google Drive checkpoint-download cell has been removed.


# Prepration

In [ ]:
# Install required packages for loading the model and inference
! pip install unsloth==2026.5.4
! pip install transformers==4.56.2 trl==0.22.2
! pip install torch==2.10.0+cu128
! pip install xformers==0.0.35
! pip install Triton==3.6.0

In [ ]:
# Import core modules for model loading and inference utilities
from unsloth import FastLanguageModel
import torch
import pandas as pd

# Model loading

In [ ]:
# Load the final LoRA + DeepSeek-R1-Distill-Llama-8B checkpoint and tokenizer
import os
model_path = os.environ.get("QA_MODEL_PATH", "/kaggle/working/fine_tuned_model")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    load_in_4bit=True,
    device_map={'': torch.cuda.current_device()},
)

In [ ]:
# Apply Unsloth's inference optimization for faster decoding (optional for HF models)
FastLanguageModel.for_inference(model)

# Model Querying

In [ ]:
# Load the test split containing context-question pairs from Drive
test_csv_path = os.environ.get('QA_TEST_CSV', '/kaggle/input/datasets/lemika/finetune-dataset-fixed-3000-new-upload/finetune_dataset_fixed_3000.csv')
test_dataset = pd.read_csv(test_csv_path)
test_dataset = test_dataset[test_dataset["split"] == "test"].reset_index(drop=True)

# Remove any pre-existing model predictions to avoid overwrite conflicts
columns_from_scoring_part = [
    'predicted_answer',
]
test_dataset.drop(columns_from_scoring_part, axis=1, inplace=True, errors='ignore')
test_dataset

In [ ]:
print(tokenizer.chat_template)

In [ ]:
# Define system prompt to enforce consistent instruction-following behavior during inference
SYSTEM_PROMPT = """You are a helpful AI assistant.

When a user provides context, use only that context to answer their question.
If the question includes multiple sub-questions, answer each one concisely on a new line, in the same order.

Always strive to provide clear, structured, and accurate answers."""

In [ ]:
# Generate model predictions using chat-style prompting and store results
model_answers = []
for index, row in test_dataset.iterrows():
    context = str(row.get("context") or "").strip()
    questions = str(row.get("question") or "").strip()
    
    # Compose chat prompt using system and user messages
    query = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context: {context}\nQuestion(s):\n{questions}" },
    ]

    prompt = tokenizer.apply_chat_template(
    query,
    tokenize=False,
    add_generation_prompt=True,
    )
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
        #padding=True,
        #tokenize=True,
        #add_generation_prompt=True,
    ).to("cuda")

    # Generate model response using constrained decoding settings
    input_length = inputs["input_ids"].shape[1]

    model.eval()
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        use_cache=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )
    
    model_answers.append(answer)
    
    # Optional: print raw model output for inspection
    print('----------------------------------------------------------')
    print(answer)

# Append predictions to test DataFrame
test_dataset['predicted_answer'] = model_answers
print('----------------------------------------------------------')
test_dataset

# Dataset saving

In [ ]:
import os
OUTPUT_DIR = os.environ.get("QA_OUTPUT_DIR", "/kaggle/working/test_result")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save predictions to Drive for evaluation or scoring
test_dataset.to_csv(os.path.join(OUTPUT_DIR, 'sampled_dataset_test_split_only.csv'), index=False)
test_dataset